[download this notebook here](https://github.com/HumanCompatibleAI/imitation/blob/master/docs/tutorials/4_train_airl.ipynb)
# Train an Agent using Adversarial Inverse Reinforcement Learning

As usual, we first need an expert. Again, we download one from the HuggingFace model hub for convenience.

Note that we now use a variant of the CartPole environment from the seals package, which has fixed episode durations. Read more about why we do this [here](https://imitation.readthedocs.io/en/latest/getting-started/variable-horizon.html).

In [1]:
import numpy as np
from imitation.policies.serialize import load_policy
from imitation.util.util import make_vec_env
from imitation.data.wrappers import RolloutInfoWrapper

SEED = 42

FAST = True

if FAST:
    N_RL_TRAIN_STEPS = 100_000
else:
    N_RL_TRAIN_STEPS = 2_000_000

venv = make_vec_env(
    "seals:seals/CartPole-v0",
    rng=np.random.default_rng(SEED),
    n_envs=8,
    post_wrappers=[
        lambda env, _: RolloutInfoWrapper(env)
    ],  # needed for computing rollouts later
)
expert = load_policy(
    "ppo-huggingface",
    organization="HumanCompatibleAI",
    env_name="seals/CartPole-v0",
    venv=venv,
)

/home/javier/Ramblings/Super-Tennis-Project/SuperTennisAI/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/javier/Ramblings/Super-Tennis-Project/SuperTennisAI/.venv/lib/python3.10/site-packages/stable_baselines3/common/save_util.py:449: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this m

We generate some expert trajectories, that the discriminator needs to distinguish from the learner's trajectories.

In [2]:
from imitation.data import rollout

rollouts = rollout.rollout(
    expert,
    venv,
    rollout.make_sample_until(min_timesteps=None, min_episodes=60),
    rng=np.random.default_rng(SEED),
)

Now we are ready to set up our AIRL trainer.
Note, that the `reward_net` is actually the network of the discriminator.
We evaluate the learner before and after training so we can see if it made any progress.

In [7]:
from imitation.algorithms.adversarial.airl import AIRL
from imitation.rewards.reward_nets import BasicShapedRewardNet
from imitation.util.networks import RunningNorm
from stable_baselines3 import PPO
from stable_baselines3.ppo import MlpPolicy
from stable_baselines3.common.evaluation import evaluate_policy


learner = PPO(
    env=venv,
    policy=MlpPolicy,
    batch_size=64,
    ent_coef=0.0,
    learning_rate=0.0005,
    gamma=0.95,
    clip_range=0.1,
    vf_coef=0.1,
    n_epochs=5,
    seed=SEED,
)
reward_net = BasicShapedRewardNet(
    observation_space=venv.observation_space,
    action_space=venv.action_space,
    normalize_input_layer=RunningNorm,
)
airl_trainer = AIRL(
    demonstrations=rollouts,
    demo_batch_size=2048,
    gen_replay_buffer_capacity=512,
    n_disc_updates_per_round=16,
    venv=venv,
    gen_algo=learner,
    reward_net=reward_net,
)

venv.seed(SEED)
learner_rewards_before_training, _ = evaluate_policy(
    learner, venv, 100, return_episode_rewards=True
)
airl_trainer.train(N_RL_TRAIN_STEPS)
venv.seed(SEED)
learner_rewards_after_training, _ = evaluate_policy(
    learner, venv, 100, return_episode_rewards=True
)

round:   0%|          | 0/122 [00:00<?, ?it/s]

------------------------------------------
| raw/                        |          |
|    gen/rollout/ep_len_mean  | 500      |
|    gen/rollout/ep_rew_mean  | 34.4     |
|    gen/time/fps             | 2582     |
|    gen/time/iterations      | 1        |
|    gen/time/time_elapsed    | 6        |
|    gen/time/total_timesteps | 16384    |
------------------------------------------
--------------------------------------------------
| raw/                                |          |
|    disc/disc_acc                    | 0.505    |
|    disc/disc_acc_expert             | 1        |
|    disc/disc_acc_gen                | 0.0103   |
|    disc/disc_entropy                | 0.663    |
|    disc/disc_loss                   | 0.74     |
|    disc/disc_proportion_expert_pred | 0.995    |
|    disc/disc_proportion_expert_true | 0.5      |
|    disc/global_step                 | 1        |
|    disc/n_expert                    | 2.05e+03 |
|    disc/n_generated                 | 2.05e+03 |
-

round:   1%|          | 1/122 [00:17<35:12, 17.45s/it]

------------------------------------------------------
| raw/                               |               |
|    gen/rollout/ep_len_mean         | 500           |
|    gen/rollout/ep_rew_mean         | 36.7          |
|    gen/rollout/ep_rew_wrapped_mean | -572          |
|    gen/time/fps                    | 2590          |
|    gen/time/iterations             | 1             |
|    gen/time/time_elapsed           | 6             |
|    gen/time/total_timesteps        | 32768         |
|    gen/train/approx_kl             | 0.00092341786 |
|    gen/train/clip_fraction         | 0.026         |
|    gen/train/clip_range            | 0.1           |
|    gen/train/entropy_loss          | -0.692        |
|    gen/train/explained_variance    | -0.014631152  |
|    gen/train/learning_rate         | 0.0005        |
|    gen/train/loss                  | 4.4           |
|    gen/train/n_updates             | 5             |
|    gen/train/policy_gradient_loss  | -0.000137     |
|    gen/t

round:   2%|▏         | 2/122 [00:35<35:04, 17.54s/it]

...

-----------------------------------------------------
| raw/                               |              |
|    gen/rollout/ep_len_mean         | 500          |
|    gen/rollout/ep_rew_mean         | 500          |
|    gen/rollout/ep_rew_wrapped_mean | -52.4        |
|    gen/time/fps                    | 2552         |
|    gen/time/iterations             | 1            |
|    gen/time/time_elapsed           | 6            |
|    gen/time/total_timesteps        | 1671168      |
|    gen/train/approx_kl             | 0.0022567455 |
|    gen/train/clip_fraction         | 0.061        |
|    gen/train/clip_range            | 0.1          |
|    gen/train/entropy_loss          | -0.157       |
|    gen/train/explained_variance    | 0.99852204   |
|    gen/train/learning_rate         | 0.0005       |
|    gen/train/loss                  | 0.0341       |
|    gen/train/n_updates             | 505          |
|    gen/train/policy_gradient_loss  | 0.00049      |
|    gen/train/value_loss   

round:  84%|████████▎ | 102/122 [28:18<05:25, 16.26s/it]

-----------------------------------------------------
| raw/                               |              |
|    gen/rollout/ep_len_mean         | 500          |
|    gen/rollout/ep_rew_mean         | 500          |
|    gen/rollout/ep_rew_wrapped_mean | -85.4        |
|    gen/time/fps                    | 2601         |
|    gen/time/iterations             | 1            |
|    gen/time/time_elapsed           | 6            |
|    gen/time/total_timesteps        | 1687552      |
|    gen/train/approx_kl             | 0.0021676589 |
|    gen/train/clip_fraction         | 0.0575       |
|    gen/train/clip_range            | 0.1          |
|    gen/train/entropy_loss          | -0.156       |
|    gen/train/explained_variance    | 0.9973787    |
|    gen/train/learning_rate         | 0.0005       |
|    gen/train/loss                  | 0.00714      |
|    gen/train/n_updates             | 510          |
|    gen/train/policy_gradient_loss  | 0.000739     |
|    gen/train/value_loss   

round:  84%|████████▍ | 103/122 [28:37<05:16, 16.67s/it]


KeyboardInterrupt: 

We can see that an untrained policy performs poorly, while AIRL brings an improvement. To make it match the expert performance (500), set the flag `FAST` to `False` in the first cell.

In [ ]:
print(
    "Rewards before training:",
    np.mean(learner_rewards_before_training),
    "+/-",
    np.std(learner_rewards_before_training),
)
print(
    "Rewards after training:",
    np.mean(learner_rewards_after_training),
    "+/-",
    np.std(learner_rewards_after_training),
)

Rewards before training: 102.6 +/- 24.11514047232568
Rewards after training: 49.43 +/- 2.2237580803675563
